## Data Parsing and API Calling

In [1]:
from datasets import load_dataset

ds = load_dataset("iamtarun/python_code_instructions_18k_alpaca")

c:\Users\denni\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\denni\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
ds["train"][0]

{'instruction': 'Create a function to calculate the sum of a sequence of integers.',
 'input': '[1, 2, 3, 4, 5]',
 'output': '# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum',
 'prompt': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate a function to calculate the sum of a sequence of integers.\n\n### Input:\n[1, 2, 3, 4, 5]\n\n### Output:\n# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum'}

In [3]:
ls = []

for i in range(len(ds["train"])):
    ls.append(ds["train"][i])

In [4]:
len(ls)
    

18612

In [5]:
import os
import json
import openai
from openai import OpenAI
from tqdm import tqdm
from pydantic import BaseModel
from typing import Optional
import time
import random
from retrying import retry
import requests
import re
from joblib import Parallel, delayed
from tqdm import tqdm

class GPT4:
    def __init__(self, your_api_key, model='gpt-4o-mini') -> None:
        self.key_ind = 0
        self.model = model
        self.url = "https://api.yesapikey.com/v1/chat/completions"
        self.key = your_api_key

        print(f'use model of {self.model}')

    def openai_call(self, content ,args = {}):
        api_key = self.key

        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        }

        parameters = {
            "model": self.model,
            "messages": [{'role': 'user', 'content': content}],
            **args,
        }

        response = requests.post(
            self.url,
            headers=headers,
            json=parameters
        )

        response = json.loads(response.content.decode("utf-8"))
        if 'error' in response:
            assert False, str(response)
        return response['choices'][0]['message']['content']

    @retry(wait_fixed=100, stop_max_attempt_number=3)
    def call(self, content, args = {}):
        return self.openai_call(content, args)

In [6]:
openai.api_key = os.getenv('OPENAI_API_KEY') 

In [7]:
use_model = 'gpt-4o-mini'

gpt = GPT4(openai.api_key,model=use_model)

use model of gpt-4o-mini


##  Structured Response

In [8]:
# Define the response model using Pydantic
class EvaluationResponse(BaseModel):
    Classification: str
    Explanation: str

In [9]:
ls

[{'instruction': 'Create a function to calculate the sum of a sequence of integers.',
  'input': '[1, 2, 3, 4, 5]',
  'output': '# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum',
  'prompt': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate a function to calculate the sum of a sequence of integers.\n\n### Input:\n[1, 2, 3, 4, 5]\n\n### Output:\n# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum'},
 {'instruction': 'Generate a Python code for crawling a website for a specific type of data.',
  'input': 'website: www.example.com \ndata to crawl: phone numbers',
  'output': "import requests\nimport re\n\ndef crawl_website_for_phone_numbers(website):\n    response = requests.get(website)\n    phone_numbers = re.findall('\\d{3}-\\d{3}-\\d{4}', response.text)\n    return phone_numbers\n    \ni

## Prompt Engineering and Output Parser

In [10]:
def evaluate_sample(sample):
    prompt = f'''
        You are a Data Quality Evaluator tasked with filtering a dataset of Python coding examples to select only the highest quality data for training a language model. For each data sample, you will be provided with:

        - **Instruction**: A description of the coding task the model should perform.
        - **Input**: The input data for the task (may be an empty list, value, or string).
        - **Output**: The Python code generated by a language model to solve the task.

        **Your task is to evaluate the following data sample according to the following criteria:**

        1. **Relevance**: Does the output directly and fully address the instruction and input?
        2. **Correctness**: Is the Python code correct? Does it solve the problem as specified in the instruction and handle edge cases?
        3. **Clarity**: Is the code easy to understand? Are the variables named appropriately? Are there clear comments explaining the steps if needed?
        4. **Efficiency**: Does the code implement the solution in an efficient manner, both in terms of time and space complexity?
        5. **Readability**: Is the code properly formatted (e.g., indentation, spacing) and follows common Python coding conventions (e.g., PEP8)?
        6. **Completeness**: Does the code fully implement the functionality as per the instruction? Does it handle different input cases, including edge cases?
        7. **Testing**: Does the code include test cases or handle validation of input/output?

        **Example:**

        Instruction:

        "Create a function to calculate the sum of a sequence of integers."

        Input:

        "[1, 2, 3, 4, 5]"

        Output:

        ```python
        def sum_sequence(sequence):
            sum = 0
            for num in sequence:
                sum += num
            return sum
        ```

        Evaluation:

        ```json
        {{
            "Classification": "High Quality",
            "Explanation": "The output is relevant and directly solves the problem (Relevance). The code is correct and implements the sum of integers function (Correctness). It is clear and easy to understand with proper variable names (Clarity), follows standard Python formatting (Readability), and provides a complete solution (Completeness)."
        }}
        ```

        **Here is the data sample to evaluate:**

        **Instruction:**

        {sample['instruction']}

        **Input:**

        {sample['input'] if sample['input'] else '""'}

        **Output (Python Code):**

        {sample['output']}

        **Please evaluate the sample and provide your evaluation.**

        Respond ONLY AND ALWAYS in JSON format that matches the following schema:

        ```json
        {{
            "Classification": "High Quality" or "Low Quality",
            "Explanation": "Provide a brief explanation citing specific criteria."
        }}
        ```
    '''

    try:
        response = gpt.call(prompt)
        json_match = re.search(r'```json\n(.*?)\n```', response, re.DOTALL)
        if json_match:
            json_str = json_match.group(1)
            # Parse the JSON string
            json_data = json.loads(json_str)
            # Validate the parsed JSON data
            evaluation = EvaluationResponse.model_validate(json_data)
            return evaluation
        else:
            print(response)
            raise ValueError("No JSON found in the response")
        
    except Exception as e:
        print(f"Error evaluating sample: {e}")
        return None

## Evaluation Process

In [11]:
# The process_sample function that uses evaluate_sample
def process_sample(sample):
    evaluation = evaluate_sample(sample)
    if evaluation and evaluation.Classification == 'High Quality':
        sample['evaluation'] = evaluation.model_dump()  # Store evaluation in the sample
        return evaluation, sample
    return evaluation, None  # Return None if the sample is not high quality

# Parallelize the evaluation process
results = Parallel(n_jobs=-1, backend="threading")(delayed(process_sample)(sample) for sample in tqdm(ls, desc="Evaluating samples"))

# Filter out high-quality data
high_quality_data = [sample for evaluation, sample in results if sample]
evaluations = [evaluation for evaluation, sample in results if evaluation]


Evaluating samples:   4%|▍         | 832/18612 [01:55<42:18,  7.00it/s]

Error evaluating sample: Expecting ',' delimiter: line 3 column 339 (char 377)


Evaluating samples:  12%|█▏        | 2160/18612 [05:18<42:52,  6.39it/s]  

Error evaluating sample: Expecting ',' delimiter: line 3 column 153 (char 192)


Evaluating samples:  21%|██        | 3872/18612 [09:36<38:14,  6.43it/s]

...

Evaluating samples:  95%|█████████▌| 17728/18612 [43:14<02:10,  6.75it/s]

Error evaluating sample: Expecting ',' delimiter: line 3 column 724 (char 763)


Evaluating samples:  96%|█████████▌| 17904/18612 [43:39<01:37,  7.23it/s]

Error evaluating sample: Expecting ',' delimiter: line 3 column 767 (char 806)


Evaluating samples: 100%|██████████| 18612/18612 [45:21<00:00,  6.84it/s]


In [44]:
with open('python_gpt4_high_quality_data.json', 'w') as f:
    json.dump(high_quality_data, f, indent=2)

print(f"Total high-quality samples: {len(high_quality_data)}")


Total high-quality samples: 6991


In [30]:
len(evaluations)

18580

In [45]:
evaluations_dict = [evaluation.dict() for evaluation in evaluations]

with open('python_gpt4_evaluation.json', 'w') as f:
    json.dump(evaluations_dict, f, indent=2)

## Result of Evaluations

In [47]:
with open('python_gpt4_high_quality_data_with_eval.json', 'w') as f:
    json.dump(high_quality_data, f, indent=2)

print(len(high_quality_data))

6991


In [49]:
import copy

high_quality_data_wo_eval =  copy.deepcopy(high_quality_data)

for item in high_quality_data_wo_eval:
    if "evaluation" in item:
        del item["evaluation"]

print(len(high_quality_data_wo_eval))


6991


In [ ]:
with open('python_gpt4_high_quality_data.json', 'w') as f:
    json.dump(high_quality_data_wo_eval, f, indent=2)



In [67]:
# Get the data that is low quality

low_quality_data = [sample for sample in ls if sample not in high_quality_data]


In [68]:
len(low_quality_data)

11621

In [ ]:
with open('python_gpt4_low_quality_data.json', 'w') as f:
    json.dump(low_quality_data, f, indent=2)